In [23]:
import pandas as pd
import numpy as np
import math

df =pd.read_csv('val_10.csv')


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748078 entries, 0 to 748077
Data columns (total 27 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Card Number             748078 non-null  int64  
 1   Transaction Date        748078 non-null  object 
 2   Product Category        748078 non-null  object 
 3   Merchant Type           748078 non-null  object 
 4   Merchant                748078 non-null  object 
 5   Currency                748078 non-null  object 
 6   Country                 748078 non-null  object 
 7   City                    748078 non-null  object 
 8   City Size               748078 non-null  object 
 9   Card Type               748078 non-null  object 
 10  Card Present            748078 non-null  int64  
 11  Channel                 748078 non-null  object 
 12  Device Fingerprint      748078 non-null  object 
 13  Distance From Home      748078 non-null  int64  
 14  High Risk Merchant  

### TIẾN HÀNH THỰC HIỆN FEATURE ENGINEERING TRÊN TẬP DỮ LIỆU TRAIN, TEST, VAL

In [ ]:
df['Is_High_Risk_Hour'] = df['Transaction Hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)
# 2. Tạo biến Night_Mismatch
df['Night_Mismatch'] = ((df['Is_High_Risk_Hour'] == 1) & (df['Is_IP_Country_Mismatch'] == 1)).astype(int)


In [26]:
# Gian lận ban đêm thường có giá trị khác biệt
df['Night_Amount_Log'] = df['Is_High_Risk_Hour'] * df['Amount USD']

In [ ]:
# Định nghĩa các nền tảng nguy hiểm dựa trên Heatmap
hazard_platforms = ['Chip Reader', 'Magnetic Stripe', 'NFC Payment']

# Tạo Feature: Kết hợp điều kiện POS và Nền tảng vật lý
df['is_physical_pos_hazard'] = (
    (df['Channel'].str.lower() == 'pos') & 
    (df['Platform Used'].isin(hazard_platforms))
).astype(int)



is_physical_pos_hazard
0    683123
1     64955
Name: count, dtype: int64


In [ ]:
# [Risk Ratio] Regional IP Mismatch Target Encoding ---
# Tính toán tỉ lệ gian lận cho từng cặp (Region Type, Is_IP_Country_Mismatch)
region_ip_map = df.groupby(['Region Type', 'Is_IP_Country_Mismatch'])['Is Fraudulent'].mean().to_dict()
df['region_ip_risk_score'] = df.set_index(['Region Type', 'Is_IP_Country_Mismatch']).index.map(region_ip_map)
# Đếm số lượng khách hàng duy nhất trên mỗi thiết bị
# Đếm số thẻ duy nhất trên mỗi thiết bị
device_card_map = df.groupby('Device Fingerprint')['Card Number'].nunique()
df['card_density_per_device'] = df['Device Fingerprint'].map(device_card_map)
# [Velocity Frequency Encoding] Mã hóa tần suất giao dịch của thiết bị ---
# Đếm số lượng giao dịch phát sinh từ cùng một thiết bị
device_freq_map = df.groupby('Device Fingerprint').size()
df['device_transaction_count_1h'] = df['Device Fingerprint'].map(device_freq_map)
# [Merchant Risk] Target Encoding cho Merchant Type ---
# Thay tên loại hình cửa hàng bằng tỉ lệ gian lận lịch sử
merchant_risk_map = df.groupby('Merchant Type')['Is Fraudulent'].mean()
df['merchant_type_fraud_rate'] = df['Merchant Type'].map(merchant_risk_map)
# [Geographical Distance Anomaly] Độ lệch khoảng cách theo khu vực ---
# Nhân khoảng cách với trọng số rủi ro vùng (International rủi ro gấp đôi Domestic)
df['dist_international_ratio'] = df['Distance From Home'] * np.where(df['Region Type'] == 'International', 1.0, 0.5)
travel_categories = ['airlines', 'hotels', 'booking']
# Tạo biến Binary: 1 nếu thuộc nhóm du lịch, 0 nếu không
df['is_travel_related'] = df['Merchant Type'].str.lower().isin(travel_categories).astype(int)
# Tạo biến tương tác: Nếu là du lịch VÀ số tiền lớn (ví dụ trên mức trung bình chung)
avg_amount = df['Amount USD'].median()
df['high_value_travel'] = ((df['is_travel_related'] == 1) & (df['Amount USD'] > avg_amount)).astype(int)
# Dọn dẹp dữ liệu sau khi đã feature engineering
cols_to_drop = [
    'Card Number', 
    'IP Address', 
    'IP_Country',
    'Merchant Type',
    'Region Type',
    'Channel',
    'Platform Used',
    'Transaction Hour' # Nếu đã có biến Is_High_Risk_Hour và Night_Mismatch
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])



In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748078 entries, 0 to 748077
Data columns (total 31 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Transaction Date             748078 non-null  object 
 1   Product Category             748078 non-null  object 
 2   Merchant                     748078 non-null  object 
 3   Currency                     748078 non-null  object 
 4   Country                      748078 non-null  object 
 5   City                         748078 non-null  object 
 6   City Size                    748078 non-null  object 
 7   Card Type                    748078 non-null  object 
 8   Card Present                 748078 non-null  int64  
 9   Device Fingerprint           748078 non-null  object 
 10  Distance From Home           748078 non-null  int64  
 11  High Risk Merchant           748078 non-null  int64  
 12  Weekend Transaction          748078 non-null  int64  
 13 

In [29]:
output_path= 'engineered_val.csv'
df.to_csv(output_path, index=False)
